## 2.Take the Institution name as input. Use Pydantic to define the schema for the desired output and create a custom output parser. Invoke the Chain and Fetch Results. Extract the below Institution related details from Wikipedia: The founder of the Institution. When it was founded. The current branches in the institution . How many employees are working in it. A brief 4-line summary of the institution.

In [28]:
# %pip install -q wikipedia-api pydantic

In [29]:
from pydantic import BaseModel
import wikipediaapi

In [30]:
class InstitutionDetails(BaseModel):
    name: str
    founder: str
    founded_year: str
    branches: str
    employees: str
    summary: str

In [33]:
def fetch_institution_details(institution_name) -> InstitutionDetails:
    wiki = wikipediaapi.Wikipedia(
        user_agent="InstitutionInfoBot/1.0 (student project)",
        language="en"
    )

    page = wiki.page(institution_name)

    if not page.exists():
        raise ValueError("Institution page does not exist on Wikipedia")

    summary = " ".join(page.summary.split(".")[:4]) + "."

    founder = "Not Available"
    founded_year = "Not Available"
    branches = "Not Available"
    employees = "Not Available"

    for section in page.sections:
        title = section.title.lower()
        if "founder" in title:
            founder = section.text.split(". ")[0]
        elif "founded" in title:
            founded_year = section.text.split(". ")[0]
        elif "branch" in title:
            branches = section.text.split(". ")[0]
        elif "employee" in title:
            employees = section.text.split(". ")[0]

    return InstitutionDetails(
        name=institution_name,
        founder=founder,
        founded_year=founded_year,
        branches=branches,
        employees=employees,
        summary=summary
    )

In [32]:
institution_name = input("Enter Institution Name: ")

try:
    details = fetch_institution_details(institution_name)
    print(details.model_dump_json(indent=4))
except ValueError as e:
    print(e)

{
    "name": "Harvard University",
    "founder": "Not Available",
    "founded_year": "Not Available",
    "branches": "Not Available",
    "employees": "Not Available",
    "summary": "Harvard University is a private Ivy League research university in Cambridge, Massachusetts, United States  Founded in 1636, and named Harvard College in 1639 in honor of its first benefactor, Puritan clergyman John Harvard, it is the oldest institution of higher learning in the United States  Its influence, wealth, and rankings have made it one of the most prestigious universities in the world \nHarvard was founded and authorized by the Massachusetts General Court, the governing legislature of colonial-era Massachusetts Bay Colony."
}
